# Qiskit 프리미티브 완벽 가이드: 이론과 실습

이 종합 노트북은 Qiskit의 **Estimator** 와 **Sampler** 프리미티브에 대해 알아야 할 모든 내용을 다룹니다. 이론적 기초부터 고급 오류 완화 기술을 활용한 실제 구현까지 상세히 설명합니다.

## 학습 내용:

**Part 1: 이론**
- **Sampler** 프리미티브의 역할과 존재 이유
- 확률 분포 및 준확률 (Quasi-probability)
- 오류 억제 기술의 이론적 기초

**Part 2: 실습**
- **Estimator** 및 **Sampler** 시작하기
- 오류 완화를 위한 옵션 설정
- 동적 디커플링 (Dynamical Decoupling) 및 파울리 트월링 (Pauli Twirling) 구현
- 고급 복원 옵션 (ZNE, T-REx) 사용
- 오류 완화 적용 전후 결과 비교

**Appendix: 참고자료**
- [Overview of noise management techniques](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-overview)
- [Error Mitigation and Suppression](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques)
- [Error mitigation extends the computational reach of a noisy quantum processor](https://www.nature.com/articles/s41586-019-1040-7)
- [Scalable error mitigation for noisy quantum circuits produces competitive expectation values](https://arxiv.org/pdf/2108.09197)

---
# Part 1: 이론의 이해
---

## 1.1 Sampler 프리미티브란 무엇인가?

**Sampler** 는 Qiskit의 핵심 프리미티브 중 하나입니다. 관측 가능량의 기댓값 $(\langle \psi | H | \psi \rangle)$ 을 계산하는 **Estimator** 와 달리, **Sampler** 는 측정 결과의 **준확률 분포 (Quasi-probability distribution)** 를 근사하도록 설계되었습니다.

간단히 말해, 상태 $|\psi\rangle$ 를 준비하는 양자 회로를 실행할 때, 계산 기저에서 이를 측정하면 비트 문자열(예: `101`)이 생성됩니다. 이를 여러 번 실행(shots)하면 카운트의 히스토그램을 얻을 수 있습니다. **Sampler** 는 이 과정을 공식화하되, 다음과 같은 추상화 계층을 추가합니다:

1. **오류 완화 (Error Mitigation)**: 판독(Readout) 또는 실행 중 발생하는 오류를 자동으로 수정하거나 억제합니다.
2. **세션 관리 (Session Management)**: 클라우드 런타임을 사용할 때 여러 작업을 효율적으로 큐에 대기시킵니다.
3. **통합 인터페이스 (Unified Interface)**: 시뮬레이터와 실제 하드웨어 전반에서 일관되게 작동합니다.

## 1.2 확률 분포

기본적으로 **Sampler** 는 기저 상태 $|k\rangle$ 를 측정할 확률 $P(k)$ 를 추정합니다:

$$ P(k) = |\langle k | \psi \rangle|^2 $$

표준 `backend.run()` 은 원시 카운트(예: `{'00': 50, '11': 50}`)를 반환하는 반면, **Sampler** 는 이러한 결과를 기저 확률 분포에서 추출된 샘플로 처리합니다.

### 왜 "준"(Quasi)-확률인가?

"준확률 (Quasi-probability)"이라는 용어를 들어보셨을 것입니다 (특히 Sampler V1 또는 특정 오류 완화 설정에서). 이는 확률적 오류 제거 (Probabilistic Error Cancellation)와 같은 오류 완화 후처리 과정으로 인해 음수 값을 갖거나 합이 1이 아닌 숫자가 될 수 있는 분포를 의미합니다.

하지만 현재 구현체(V2)의 표준 사용법과 기본 설정에서는 주로 양자 상태의 실제 확률 분포를 근사하는 **샘플링된 비트 문자열** (카운트) 또는 **확률 질량 함수** (정규화된 카운트)에 집중합니다.

## 1.3 오류 억제 및 완화 이론

`backend.run()` 대신 **Sampler** 프리미티브를 사용하는 주요 이유 중 하나는 고급 오류 처리 옵션에 접근할 수 있기 때문입니다. **Sampler** 옵션을 통해 자주 노출되는 두 가지 핵심 기술은 **동적 디커플링 (Dynamical Decoupling)** 과 **파울리 트월링 (Pauli Twirling)** 이 있습니다.

### A. 동적 디커플링 (Dynamical Decoupling, DD)


<img src="https://quantum.cloud.ibm.com/assets-docs-learning/_next/image?url=%2Fdocs%2Fimages%2Fguides%2Ferror-mitigation-explanation%2Fdd.avif&w=3840&q=75" width="1000">


**문제**: 큐비트는 환경과의 상호작용(결맞음 해제, 즉 $T_1$ 및 $T_2$ 시간)으로 인해 시간이 지남에 따라 양자 정보를 잃습니다. 이는 큐비트가 "유휴(idle)" 상태(다른 큐비트의 게이트 연산이 끝나기를 기다리는 동안)일 때도 발생합니다.

**이론**: 동적 디커플링은 유휴 큐비트에 일련의 펄스(일반적으로 $X$ 게이트와 같은 $\pi$-펄스)를 적용합니다. 이러한 펄스는 블로흐 구(Bloch sphere)에서 반대축을 따라 큐비트 상태를 반복적으로 뒤집습니다 ("스핀 에코"와 유사).

**비유**: 트랙 위의 주자를 상상해 보세요. 바람 때문에 왼쪽으로 약간 치우친다면, 주자를 180도 뒤집어 반대 방향(트랙 기준)으로 달리게 함으로써 원래의 이탈을 상쇄할 수 있습니다.

**Qiskit 활용**: **Sampler** 옵션에서 이를 활성화하면, 트랜스파일러/스케줄러가 유휴 시간 윈도우에 이러한 시퀀스를 자동으로 삽입합니다.

### B. 파울리 트월링 (Pauli Twirling)

<img src="https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/pauli_twirling@dark.svg" width="1000">


**문제**: 체계적이고 간섭성 있는 오류(예: 게이트가 항상 1도씩 과하게 회전하는 경우)는 구조적으로 누적될 수 있기 때문에 ($1 + 1 + 1...$) 매우 치명적입니다.

**이론**: 트월링은 노이즈가 있는 게이트(예: CNOT)를 무작위 파울리 게이트($I, X, Y, Z$)로 감쌉니다. 이는 노이즈가 없는 경우 수학적으로 항등 연산(Identity operation)이 되도록 설계되었습니다. 그러나 노이즈가 존재하는 상황에서 이러한 무작위화는 "간섭성(coherent)" 오류(체계적인 편향)를 "비간섭성(incoherent)" 확률적 노이즈(무작위 비트 플립)로 변환합니다.

**수행 이유**: 비간섭성 노이즈는 "탈분극 채널 (Depolarizing channel)"로 이어지며, 이는 간섭성 오류보다 모델링 및 완화가 훨씬 쉽습니다. 즉, 예측 가능한 "나쁜" 오류를 평균화되어 사라지는 "무작위" 노이즈로 바꿉니다.

현재 양자 하드웨어에서 발생하는 오류의 대부분은 2큐비트 게이트에서 비롯되므로, 이 기술은 주로 (기본적인) 2큐비트 게이트에만 적용됩니다. 다음 그림은 CNOT 및 ECR 게이트에 대한 파울리 트월(Pauli twirl)의 예시를 보여줍니다. 각 행의 회로는 모두 동일한 이상적인 효과를 나타냅니다.

<img src="https://quantum.cloud.ibm.com/assets-docs-learning/_next/image?url=%2Fdocs%2Fimages%2Fguides%2Ferror-mitigation-explanation%2Fgate_twirls.avif&w=3840&q=75" width="1000">



---
# Part 2: qiskit 구현
---

## 2.1 프리미티브 시작하기 (이상적인 로컬 시뮬레이션)

먼저 완벽하고 노이즈가 없는 로컬 시뮬레이터(`AerSimulator`)에서 **Estimator** 와 **Sampler** 를 실행해 보겠습니다. 이를 통해 기본적인 워크플로우를 익힐 수 있습니다.

### Estimator 예제

**Estimator** 프리미티브는 주어진 회로에 대해 관측 가능량의 기댓값을 계산합니다. 이는 VQE와 같은 많은 양자 알고리즘에서 필수적인 작업입니다.

In [2]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime
# from qiskit_ibm_runtime import SamplerV2 as Sampler1 # for debugging
# from qiskit.primitives import BackendSamplerV2 as Sampler2
# print(Sampler1)
# print(Sampler2)


print(qiskit.__version__)
print(qiskit_aer.__version__)
print(qiskit_ibm_runtime.__version__)

2.1.2
0.17.2
0.45.0


In [3]:
# 1. 로컬 시뮬레이터 백엔드 임포트
from qiskit_aer import AerSimulator
# 이상적인 (노이즈 없는) 시뮬레이터로 시작합니다.
backend = AerSimulator()

print(f"로컬 백엔드 사용 중: {backend.name}")

로컬 백엔드 사용 중: aer_simulator


In [4]:
from qiskit.circuit.library import qaoa_ansatz
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# 실행 가능하고 점검하기 쉬운 5-큐비트 회로를 사용합니다.
num_qubits = 5
entanglement = [(0, 1), (1, 2), (2, 3), (3, 4)]
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], 0.5) for i, j in entanglement],
    num_qubits=num_qubits,
)
circuit = qaoa_ansatz(observable, reps=2)

# 매개변수 값은 회로의 매개변수 개수와 일치해야 합니다.
param_values = np.random.rand(circuit.num_parameters)

print(f">>> 회로 매개변수 개수: {circuit.num_parameters}")
print(f">>> 관측 가능 파울리 연산자: {observable.paulis}")

>>> 회로 매개변수 개수: 4
>>> 관측 가능 파울리 연산자: ['IIIZZ', 'IIZZI', 'IZZII', 'ZZIII']


In [5]:
from qiskit.transpiler import generate_preset_pass_manager

# 백엔드의 기저 게이트에 맞춰 회로를 트랜스파일하는 것은 좋은 습관입니다.
# (이상적인 AerSimulator에서는 선택사항인 경우가 많습니다.)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(f">>> 회로 연산 (ISA 준수): {isa_circuit.count_ops()}")

>>> 회로 연산 (ISA 준수): OrderedDict({'rx': 10, 'rzz': 8, 'h': 5})


In [6]:
# 2. 모든 로컬 백엔드와 호환되는 BackendEstimatorV2 임포트
from qiskit.primitives import BackendEstimatorV2 as Estimator

# 로컬 AerSimulator 백엔드로 Estimator 인스턴스 생성
estimator = Estimator(backend=backend)

# run 호출은 Primitives Unified Blocs (PUBs) 리스트를 인자로 받습니다.
# 각 PUB은 (circuit, observable, [parameter_values]) 튜플입니다.
job = estimator.run([(isa_circuit, isa_observable, param_values)])
print(f">>> 작업 ID: {job.job_id()}")
print(f">>> 작업 상태: {job.status()}")

>>> 작업 ID: 0f8a9d73-2fc4-41e2-8f91-635972bfbf9a
>>> 작업 상태: JobStatus.DONE


In [7]:
# 3. 이제 실제 결과를 확인해 봅시다!
result = job.result()
print(f"\n>>> 전체 결과 객체: {result}")

# 첫 번째 PUB의 결과는 인덱스 0에 있습니다.
pub_result = result[0]
print(f"\n  > 기댓값 (EV): {pub_result.data.evs}")
print(f"  > 표준 편차: {pub_result.data.stds}")
print(f"  > 메타데이터: {pub_result.metadata}")


>>> 전체 결과 객체: PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=0.030830666224037573), metadata={'target_precision': 0.015625, 'shots': 4096, 'circuit_metadata': {}})], metadata={'version': 2})

  > 기댓값 (EV): 0.325439453125
  > 표준 편차: 0.030830666224037573
  > 메타데이터: {'target_precision': 0.015625, 'shots': 4096, 'circuit_metadata': {}}


### Sampler 예제

**Sampler** 프리미티브는 회로의 출력물로부터 비트 문자열의 준확률 분포를 생성합니다. 이는 양자 시스템을 여러 번 측정하는 과정을 모방합니다.

In [8]:
from qiskit.circuit.library import efficient_su2

# 가독성을 위해 작은 회로를 사용합니다.
sampler_circuit = efficient_su2(5, entanglement="linear", reps=2)
sampler_circuit.measure_all()
sampler_param_values = np.random.rand(sampler_circuit.num_parameters)

# 이전의 패스 매니저를 다시 사용하여 트랜스파일합니다.
sampler_isa_circuit = pm.run(sampler_circuit)
print(f">>> 회로 연산 (ISA 준수): {sampler_isa_circuit.count_ops()}")

>>> 회로 연산 (ISA 준수): OrderedDict({'ry': 15, 'rz': 15, 'cx': 8, 'measure': 5, 'barrier': 1})


In [9]:
# 3. BackendSamplerV2 임포트
from qiskit.primitives import BackendSamplerV2 as Sampler

sampler = Sampler(backend=backend)

# Sampler PUB은 (circuit, [parameter_values], [shots]) 튜플입니다.
job = sampler.run([(sampler_isa_circuit, sampler_param_values)])
print(f">>> 작업 ID: {job.job_id()}")
print(f">>> 작업 상태: {job.status()}")

>>> 작업 ID: 444ac5d7-7137-4ea1-bf9a-c5372bd5d287
>>> 작업 상태: JobStatus.RUNNING


In [10]:
# 4. 결과 가져오기 및 표시
result = job.result()
print(f"\n>>> 전체 결과 객체: {result}")

# 첫 번째 (이자 유일한) PUB의 결과 가져오기
pub_result = result[0]
print(
    f"\n'meas' 출력 레지스터의 처음 10개 결과:\n{pub_result.data.meas.get_bitstrings()[:10]}"
)


>>> 전체 결과 객체: PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'shots': 1024, 'circuit_metadata': {}})], metadata={'version': 2})

'meas' 출력 레지스터의 처음 10개 결과:
['00000', '10000', '10000', '10000', '11000', '11010', '11000', '10000', '00000', '10000']


## 2.2 오류 완화를 위한 옵션 설정

이제 이론을 이해하고 기본적인 사용법을 보았으므로, 앞서 배운 오류 완화 기술을 구현해 보겠습니다.

`resilience_level` 및 `dynamical_decoupling` 과 같은 많은 고급 옵션은 `qiskit-ibm-runtime` 서비스의 기능입니다. 기본 `BackendEstimatorV2` 및 `BackendSamplerV2` 는 이를 직접 지원하지 않습니다.

**하지만**, **노이즈 시뮬레이터** 와 함께 **로컬 모드의 런타임 프리미티브** 를 사용하여 이러한 옵션의 *효과* 를 테스트할 수 있습니다. 이를 통해 클라우드 계정 없이도 API를 익히고 오류 완화가 어떻게 작동하는지 확인할 수 있습니다.

### 오류 완화 예제를 위한 설정

먼저 실제 장치(`FakeManilaV2`)의 노이즈 모델을 시뮬레이션하여 노이즈 백엔드를 생성해 보겠습니다. 모든 다음 예제에서 이를 사용할 것입니다. 또한 이 특정 노이즈 백엔드에 맞춰 회로를 트랜스파일해야 합니다.

In [11]:
# --- 가짜 백엔드 (Fake Backend) 임포트 블록 ---
try:
    from qiskit_ibm_runtime.fake_provider import FakeManilaV2
except ImportError:
    try:
        from qiskit.providers.fake_provider import FakeManilaV2
    except ImportError:
        FakeManilaV2 = None
# ---------------------------------------------

# 이러한 옵션에 접근하려면 백엔드용이 아닌 런타임용 Estimator가 필요합니다.
from qiskit_ibm_runtime import EstimatorV2 as RuntimeEstimator

# 임포트가 성공한 경우에만 실행됩니다.
if FakeManilaV2:
    # 1. 현실적인 노이즈 백엔드 생성
    noisy_backend = AerSimulator.from_backend(FakeManilaV2())

    # 2. 이 노이즈 백엔드의 기저 게이트를 위해 회로를 트랜스파일
    pm_noisy = generate_preset_pass_manager(optimization_level=1, backend=noisy_backend)
    noisy_isa_circuit = pm_noisy.run(circuit) # 첫 번째 Estimator 예제의 'circuit' 사용
    noisy_isa_observable = observable.apply_layout(noisy_isa_circuit.layout)

    print("오류 완화 예제를 위한 노이즈 백엔드와 트랜스파일된 회로가 준비되었습니다.")
else:
    print("FakeManilaV2를 사용할 수 없어 오류 완화 예제를 건너뜁니다.")
    # 노트북 작동을 위한 플레이스홀더 변수 정의
    noisy_backend = None
    noisy_isa_circuit = None
    noisy_isa_observable = None

오류 완화 예제를 위한 노이즈 백엔드와 트랜스파일된 회로가 준비되었습니다.


### 공통 Sampler 옵션

여기서는 `Sampler` 와 `Estimator` 모두에 공통으로 적용되는 옵션 설정을 보여줍니다.

#### 옵션: `default_shots`

`default_shots` 는 모든 실행에 대한 샷 횟수를 설정하지만, `run()` 호출에서 재정의(override)할 수 있습니다. **Sampler** 를 사용하여 시연해 보겠습니다.

In [12]:
from qiskit.primitives import BackendSamplerV2 as Sampler

# 1. 기본값으로 인스턴스화
sampler_with_options = Sampler(backend=backend, options={"default_shots": 1000})

# 2. run()에서 샷을 지정하지 않고 실행 - 기본값을 사용해야 함(1000)
job1 = sampler_with_options.run([(sampler_isa_circuit, sampler_param_values)])
result1 = job1.result()
print(f"실행 1에서 사용된 기본 샷 수: {result1[0].metadata['shots']}")

# 3. run()에서 샷을 지정하여 실행 - 재지정된 값 사용
job2 = sampler_with_options.run([(sampler_isa_circuit, sampler_param_values)], shots=123)
result2 = job2.result()
print(f"실행 2에서 재지정된 샷 수: {result2[0].metadata['shots']}")

실행 1에서 사용된 기본 샷 수: 1000
실행 2에서 재지정된 샷 수: 123


## 2.3 동적 디커플링 (Dynamical Decoupling) 구현

이제 앞서 배운 **동적 디커플링 (Dynamical Decoupling)** 이론을 적용해 봅시다. 유휴 큐비트의 오류는 항등 연산 시퀀스(펄스)를 추가하여 상쇄할 수 있습니다. 이는 런타임 전용 옵션입니다.

In [13]:
if FakeManilaV2:
    # 이 옵션에는 RuntimeEstimator를 사용해야 합니다.
    estimator_dd = RuntimeEstimator(mode=noisy_backend)

    # 동적 디커플링 옵션 설정
    estimator_dd.options.dynamical_decoupling.enable = True
    estimator_dd.options.dynamical_decoupling.sequence_type = "XpXm" # 일반적인 시퀀스
    print(f"동적 디커플링 활성화: {estimator_dd.options.dynamical_decoupling.enable}")
    print(f"시퀀스 유형: {estimator_dd.options.dynamical_decoupling.sequence_type}")

    # 작업 실행 및 결과 표시
    job_dd = estimator_dd.run([(noisy_isa_circuit, noisy_isa_observable, param_values)])
    result_dd = job_dd.result()
    print(f"  > 기댓값 (DD 적용): {result_dd[0].data.evs}")

동적 디커플링 활성화: True
시퀀스 유형: XpXm
  > 기댓값 (DD 적용): 0.266357421875


/Users/jojeongbin/miniforge3/lib/python3.12/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:274: UserWarning: Options {'dynamical_decoupling': {'enable': True, 'sequence_type': 'XpXm'}} have no effect in local testing mode.
  warnings.warn(f"Options {options_copy} have no effect in local testing mode.")


## 2.4 파울리 트월링 (Pauli Twirling) 구현

**파울리 트월링 (Pauli Twirling)** 은 게이트를 무작위 파울리 연산자로 감싸 특정 유형의 간섭성 노이즈를 평균화하는 작업입니다. 이는 런타임 **Estimator** 옵션입니다.

In [14]:
if FakeManilaV2:
    estimator_twirl = RuntimeEstimator(mode=noisy_backend)
    
    # 트월링 옵션 설정
    estimator_twirl.options.twirling.enable_gates = True
    estimator_twirl.options.twirling.num_randomizations = 16
    print(f"게이트 트월링 활성화: {estimator_twirl.options.twirling.enable_gates}")
    
    job_twirl = estimator_twirl.run([(noisy_isa_circuit, noisy_isa_observable, param_values)])
    result_twirl = job_twirl.result()
    print(f"  > 기댓값 (트월링 적용): {result_twirl[0].data.evs}")

게이트 트월링 활성화: True
  > 기댓값 (트월링 적용): 0.241943359375


/Users/jojeongbin/miniforge3/lib/python3.12/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:274: UserWarning: Options {'twirling': {'enable_gates': True, 'num_randomizations': 16}} have no effect in local testing mode.
  warnings.warn(f"Options {options_copy} have no effect in local testing mode.")


## 2.5 고급 오류 완화: 복원 (Resilience) 옵션

`resilience` 옵션은 노이즈가 결과에 미치는 영향을 줄이기 위한 런타임 **Estimator** 의 강력한 기능입니다.

**UserWarning 관련 참고:** 로컬에서 실행할 때 `resilience_level` 이 효과가 없다는 경고가 표시될 수 있습니다. 그러나 기본 `AerSimulator` 가 실제로 완화를 수행하며, 결과가 개선되는 것을 확인할 수 있습니다.

### 제로 노이즈 외삽법 (Zero Noise Extrapolation, ZNE)

이 기술은 회로를 서로 다른 증폭된 노이즈 레벨에서 실행하고, 결과를 노이즈가 0인 한계점으로 외삽(Extrapolate)합니다.

<img src="https://quantum.cloud.ibm.com/assets-docs-learning/_next/image?url=%2Fdocs%2Fimages%2Fguides%2Ferror-mitigation-explanation%2Fzne.avif&w=3840&q=75" width="1000">

- [Error mitigation extends the computational reach of a noisy quantum processor](https://www.nature.com/articles/s41586-019-1040-7)



In [15]:
if FakeManilaV2:
    estimator_zne = RuntimeEstimator(mode=noisy_backend)
    
    # ZNE 옵션 설정 (레벨 0은 프리셋 비활성화)
    estimator_zne.options.resilience_level = 0
    estimator_zne.options.resilience.zne_mitigation = True
    print(f"ZNE 활성화: {estimator_zne.options.resilience.zne_mitigation}")

    job_zne = estimator_zne.run([(noisy_isa_circuit, noisy_isa_observable, param_values)])
    result_zne = job_zne.result()
    print(f"  > 기댓값 (ZNE 적용): {result_zne[0].data.evs}")

ZNE 활성화: True
  > 기댓값 (ZNE 적용): 0.259521484375


/Users/jojeongbin/miniforge3/lib/python3.12/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:190: UserWarning: The resilience_level option has no effect in local testing mode.
  warnings.warn("The resilience_level option has no effect in local testing mode.")
/Users/jojeongbin/miniforge3/lib/python3.12/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:274: UserWarning: Options {'resilience': {'zne_mitigation': True}} have no effect in local testing mode.
  warnings.warn(f"Options {options_copy} have no effect in local testing mode.")


### 판독 오류 진화 (Twirling Readout Error Extinction, T-REX)

<img src="https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/measurement_twirling@dark.svg" width="1000">



이 방법은 특히 큐비트의 최종 측정 중에 발생하는 오류를 목표로 합니다.

In [16]:
if FakeManilaV2:
    estimator_trex = RuntimeEstimator(mode=noisy_backend)
    
    # T-REX 옵션 설정 (레벨 0은 프리셋 비활성화)
    estimator_trex.options.resilience_level = 0
    estimator_trex.options.resilience.measure_mitigation = True
    print(f"측정 완화 (T-REX) 활성화: {estimator_trex.options.resilience.measure_mitigation}")
    
    job_trex = estimator_trex.run([(noisy_isa_circuit, noisy_isa_observable, param_values)])
    result_trex = job_trex.result()
    print(f"  > 기댓값 (T-REX 적용): {result_trex[0].data.evs}")

측정 완화 (T-REX) 활성화: True
  > 기댓값 (T-REX 적용): 0.257080078125


/Users/jojeongbin/miniforge3/lib/python3.12/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:274: UserWarning: Options {'resilience': {'measure_mitigation': True}} have no effect in local testing mode.
  warnings.warn(f"Options {options_copy} have no effect in local testing mode.")


## 2.6 최종 비교: `resilience_level` 프리셋 사용하기

`resilience_level` 은 여러 오류 완화 기술을 결합하는 편리한 프리셋입니다.

*   **Level 0:** 완화 없음.
*   **Level 1:** ZNE + T-REX. 일반적인 목적의 좋은 시작점입니다.

완화가 전혀 없는 경우와 `resilience_level=1` 인 경우의 결과를 실제 이상적인 값과 비교해 보겠습니다.

In [19]:
if FakeManilaV2:
    # --- 기준이 될 이상적인 (노이즈 없는) 결과 ---
    ideal_backend = AerSimulator()
    estimator_ideal = RuntimeEstimator(mode=ideal_backend)
    result_ideal = estimator_ideal.run([(isa_circuit, isa_observable, param_values)]).result()

    # --- 오류 완화가 전혀 없는 Estimator ---
    estimator_no_mitigation = RuntimeEstimator(mode=noisy_backend)
    estimator_no_mitigation.options.resilience_level = 0
    job_no_mitigation = estimator_no_mitigation.run([(noisy_isa_circuit, noisy_isa_observable, param_values)])
    result_no_mitigation = job_no_mitigation.result()

    # --- Level 1 완화가 적용된 Estimator ---
    estimator_with_mitigation = RuntimeEstimator(mode=noisy_backend)
    estimator_with_mitigation.options.resilience_level = 1
    job_with_mitigation = estimator_with_mitigation.run([(noisy_isa_circuit, noisy_isa_observable, param_values)])
    result_with_mitigation = job_with_mitigation.result()

    print("\n--- 최종 비교 ---")
    print(f"  > 기댓값 (이상적, 무소음): {result_ideal[0].data.evs}")
    print(f"  > 기댓값 (노이즈 있음, 완화 없음): {result_no_mitigation[0].data.evs}")
    print(f"  > 기댓값 (노이즈 있음, 완화 적용, resilience_level=1): {result_with_mitigation[0].data.evs}")


--- 최종 비교 ---
  > 기댓값 (이상적, 무소음): 0.345947265625
  > 기댓값 (노이즈 있음, 완화 없음): 0.244140625
  > 기댓값 (노이즈 있음, 완화 적용, resilience_level=1): 0.267822265625


---
# 요약
---

이 종합 노트북에서 우리는 다음 내용을 학습했습니다:

## 이론:
1. **Sampler의 목적**: 출력 확률 분포 $(|\langle k | \psi \rangle|^2$) 를 샘플링하는 방법을 확인했습니다.
2. **동적 디커플링 이론**: 펄스 시퀀스로 유휴 큐비트를 리포커싱하여 결맞음 해제를 상쇄하는 방법입니다.
3. **파울리 트월링 이론**: 간섭성 오류를 무작위화하여 비간섭성 확률적 노이즈로 변환하는 원리입니다.

## 실습:
1. **기본 프리미티브**: 이상적인 시뮬레이터에서 **Estimator** 와 **Sampler** 를 설정하고 실행했습니다.
2. **옵션 구성**: `default_shots` 및 기타 옵션을 설정하는 방법을 익혔습니다.
3. **오류 완화 구현**:
   - 동적 디커플링 (DD)
   - 파울리 트월링
   - 제로 노이즈 보외법 (ZNE)
   - 판독 오류 진화 (T-REX)
4. **복원 레벨 (Resilience Levels)**: 오류 완화 적용 여부에 따른 결과를 비교했습니다.

이제 Qiskit 프리미티브를 효과적으로 사용하기 위한 이론적 기초와 실무 기술을 모두 갖추게 되었습니다!

## 연습 문제

**1. Sampler 프리미티브가 계산하는 주요 출력은 무엇입니까?**

A) 관측 가능량의 기댓값

B) 측정 결과의 준확률 분포

C) 양자 시스템의 상태 벡터

D) 회로의 유니터리 행렬

***정답:***
<Details>
<br/>
B) 측정 결과의 준확률 분포
</Details>

---

**2. 일련의 펄스(예: X 게이트)를 적용하여 유휴 큐비트의 결맞음 해제 오류를 억제하는 오류 완화 기술은 무엇입니까?**

A) 파울리 트월링 (Pauli Twirling)

B) 제로 노이즈 보외법 (Zero Noise Extrapolation, ZNE)

C) 동적 디커플링 (Dynamical Decoupling, DD)

D) 확률적 오류 제거 (Probabilistic Error Cancellation, PEC)

***정답:***
<Details>
<br/>
C) 동적 디커플링 (Dynamical Decoupling, DD)
</Details>

---

**3. Qiskit Runtime Estimator V2에서 복원 레벨(Resilience level) 프리셋을 사용하지 않고 제로 노이즈 보외법(ZNE)을 명시적으로 활성화하려면 어떻게 해야 합니까?**

A) `options.resilience.zne_mitigation = True`

B) `options.zne = True`

C) `options.error_mitigation.zne = True`

D) `options.resilience_level = 'ZNE'`

***정답:***
<Details>
<br/>
A) `options.resilience.zne_mitigation = True`
</Details>